# 📚 Designing Data-Intensive Applications
## Bölüm 7: Sharding (Parçalama)

---

> *"Açıkça, sıralı yaklaşımdan uzaklaşmalı ve bilgisayarları sınırlamamalıyız. Verilerin tanımlarını, önceliklerini ve açıklamalarını belirtmeliyiz. Prosedürleri değil, ilişkileri ifade etmeliyiz."*
>
> — Grace Murray Hopper, *Management and the Computer of the Future* (1962)

---

Bu notebook, DDIA kitabının 7. bölümünü Türkçe olarak kapsamlı biçimde açıklamaktadır. Tüm teknik terimler İngilizce olarak korunmuştur.

**Bölümün Ana Konuları:**
- Sharding nedir ve neden gereklidir?
- Sharding'in avantajları ve dezavantajları
- Multitenancy için sharding
- Key-value verinin sharding yöntemleri
- Request routing
- Secondary indexes ve sharding

## 🌐 Giriş: Distributed Database'lerde Veri Dağıtımı

Bir **distributed database** (dağıtık veritabanı), verileri node'lar (düğümler) arasında iki temel yolla dağıtır:

1. **Replication (Çoğaltma):** Aynı verinin kopyası birden fazla node'da saklanır. Bu konu Bölüm 6'da incelendi.

2. **Sharding (Parçalama):** Verinin hacmi ya da **write throughput** (yazma kapasitesi) tek bir node'un kaldırabileceği sınırı aşıyorsa, veri daha küçük **shard**'lara (parçalara) veya **partition**'lara bölünür ve farklı shard'lar farklı node'larda saklanır.

### Temel Kavram: Shard Nedir?

Her veri parçası (her record, row ya da document) tam olarak **bir** shard'a aittir. Her shard, kendi başına küçük bir veritabanıdır; ancak bazı veritabanı sistemleri aynı anda birden fazla shard'a dokunan işlemleri destekler.

### Sharding + Replication

Sharding genellikle replication ile birlikte kullanılır: her shard'ın kopyaları birden fazla node'da saklanır. Bu sayede her record tam olarak bir shard'a ait olsa da, **fault tolerance** (hata toleransı) sağlamak için birden fazla farklı node'da depolanabilir.

Bir node birden fazla shard saklayabilir. **Single-leader replication** modeli kullanıldığında:
- Her shard'ın **leader**'ı bir node'a atanır
- **Follower**'lar başka node'lara atanır
- Her node bazı shard'lar için leader, bazıları için follower olabilir
- Ancak her shard'ın yalnızca bir **leader**'ı vardır

## 📦 Terminoloji: Shard, Partition ve Diğer İsimler

Bu bölümde **shard** dediğimiz kavram, farklı yazılımlarda farklı isimlerle anılır:

| Yazılım | Kullandığı Terim |
|---------|------------------|
| **Kafka** | partition |
| **CockroachDB** | range |
| **HBase**, **TiDB** | region |
| **Couchbase** | vBucket |
| **Riak** | vnode |
| **Cassandra** | token-range |
| **Bigtable**, **YugabyteDB**, **ScyllaDB** | tablet |

### Partitioning vs. Sharding Farkı

Bazı veritabanları bu iki kavramı birbirinden farklı olarak ele alır:

- **PostgreSQL'de:** *partitioning*, büyük bir tabloyu **aynı makine** üzerindeki birkaç dosyaya bölmektir (avantajı: tüm partition'ı silmek çok hızlıdır). *Sharding* ise veri kümesini **birden fazla makineye** yayar.
- **Diğer pek çok sistemde:** partitioning, sharding ile eş anlamlıdır.

### "Sharding" Kelimesinin Kökeni

"Sharding" terimi, *Ultima Online* adlı çevrimiçi rol yapma oyunundan gelir. Oyunda büyülü bir kristal parçalanmış; her **shard** (kıymık), oyun dünyasının bir kopyasını yansıtmıştır. Terim, paralel oyun sunucularını ifade etmek için kullanılmış, ardından veritabanı dünyasına taşınmıştır. Alternatif bir teoriye göre ise **SHARD**, *System for Highly Available Replicated Data* kısaltmasıdır.

> **Önemli Not:** *Network partition* (ağ bölünmesi / netsplit) kavramıyla karıştırılmamalıdır. Network partition, node'lar arasındaki ağda yaşanan bir arıza türüdür.

## ⚖️ Sharding'in Avantajları ve Dezavantajları

### Neden Sharding Kullanılır?

Sharding'in birincil nedeni **scalability** (ölçeklenebilirlik)dir. Veri hacmi veya **write throughput** tek bir node'un kaldıramayacağı kadar büyük olduğunda, bu yükü birden fazla node'a yaymak gerekir.

- **Read throughput** problem oluşturuyorsa sharding'e gerek olmayabilir; bunun yerine Bölüm 6'da anlatılan **read scaling** (okuma ölçeklendirmesi) yeterlidir.

Sharding, **horizontal scaling** (yatay ölçeklendirme) veya **scale-out architecture**'ın temel araçlarından biridir: sistemin kapasitesini daha büyük bir makineye geçmek yerine daha fazla (ve daha küçük) makine ekleyerek artırma anlamına gelir. Eğer iş yükünü her shard'ın yaklaşık eşit pay alacağı şekilde bölebilirseniz, bu shard'ları farklı makinelere atayarak verilerini ve sorgularını paralel olarak işleyebilirsiniz.

### Ne Zaman Sharding Kullanılmamalı?

Replication hem küçük hem büyük ölçekte faydalıdır; **fault tolerance** ve çevrimdışı çalışmayı sağlar. Sharding ise ağır bir çözümdür ve büyük ölçekte anlam ifade eder. Verinin hacmi ve **write throughput** tek bir makineyle yönetilebiliyorsa (bugünün makineleri çok şey yapabilir!), sharding'den kaçınıp tek-shard'lı bir veritabanıyla devam etmek genellikle daha iyidir.

### Sharding'in Getirdiği Karmaşıklık

Sharding karmaşıklık ekler. Hangi record'ların hangi shard'a gideceğine karar vermek için bir **partition key** (bölüm anahtarı) seçmek gerekir; aynı partition key'e sahip tüm record'lar aynı shard'a yerleştirilir. Bu seçim önemlidir çünkü:

- Hangi shard'da olduğunu biliyorsanız erişim **hızlıdır**.
- Bilmiyorsanız tüm shard'larda **verimsiz bir arama** yapmak gerekir.
- Sharding düzenini sonradan değiştirmek çok **zordur**.

### Key-Value vs. Relational Veri

Sharding, **key-value** veri için genellikle iyi çalışır; key ile kolayca shard'lanabilir. **Relational** veri için daha zordur: secondary index'le arama yapmak veya farklı shard'lara dağılmış record'ları **join** etmek isteyebilirsiniz.

### Distributed Transaction Sorunu

Bir **write** işleminin birden fazla shard'daki ilgili record'ları güncellemesi gerekebilir. Tek node'daki transaction'lar oldukça yaygınken, birden fazla shard'da tutarlılığı sağlamak bir **distributed transaction** gerektirir. Bölüm 8'de görüleceği üzere, distributed transaction'lar bazı veritabanlarında mevcut olmakla birlikte genellikle tek-node transaction'larından çok daha yavaştır ve sistemin tamamında darboğaz oluşturabilir.

### Tek Makine Üzerinde Sharding

Bazı sistemler sharding'i tek makine üzerinde bile kullanır; CPU'daki paralellikten ya da **NUMA** (*nonuniform memory access*: bazı bellek bankalarının belirli CPU'lara daha yakın olduğu mimari) avantajından yararlanmak için tipik olarak her CPU core'u için tek-thread'li bir process çalıştırır. Örneğin **Redis**, **VoltDB** ve **FoundationDB**, aynı makine üzerindeki CPU core'larına yükü dağıtmak için her core için bir process ve sharding kullanır.

## 🏢 Multitenancy için Sharding

### SaaS ve Multitenancy Nedir?

**SaaS** (*Software as a Service*) ürünleri ve bulut hizmetleri genellikle **multitenant**'tır; her **tenant** bir müşteridir. Birden fazla kullanıcı aynı tenant'a ait olabilir, ancak her tenant'ın veri kümesi diğer tenant'lardan ayrı ve bağımsızdır. Örneğin bir e-posta pazarlama hizmetinde kayıt olan her işletme genellikle ayrı bir tenant'tır: bir işletmenin bülten abonelikleri, teslimat verileri vb. diğer işletmelerin verilerinden ayrıdır.

### Sharding ile Multitenancy Uygulaması

Multitenant sistemler uygulamak için bazen sharding kullanılır:
- Her tenant ayrı bir shard alır, ya da
- Birden fazla küçük tenant tek bir büyük shard'da gruplanır.

Bu shard'lar fiziksel olarak ayrı veritabanları olabilir ya da daha büyük bir mantıksal veritabanının ayrı ayrı yönetilebilir bölümleri olabilir.

### Multitenancy'de Sharding'in Avantajları

#### 1. Resource Isolation (Kaynak İzolasyonu)
Bir tenant hesaplamaca pahalı bir işlem gerçekleştirdiğinde, diğer tenant'ların performansının bundan etkilenme olasılığı, farklı shard'larda çalışıyorlarsa çok daha azdır.

#### 2. Permission Isolation (İzin İzolasyonu)
**Access control** (erişim denetimi) mantığında bir hata olsa bile, farklı tenant'ların verileri fiziksel olarak ayrı depolandığında, bir tenant'ın yanlışlıkla başka bir tenant'ın verilerine erişme ihtimali azalır.

#### 3. Cell-Based Architecture (Hücre Tabanlı Mimari)
Sharding'i yalnızca veri depolama katmanına değil, uygulama kodunu çalıştıran servislere de uygulamak mümkündür. **Cell-based architecture**'da, belirli bir tenant grubunun servisleri ve depolama alanı bağımsız bir **cell** (hücre) içinde gruplanır; farklı cell'ler birbirinden büyük ölçüde bağımsız çalışacak şekilde kurulur. Bu yaklaşım **fault isolation** (hata izolasyonu) sağlar: bir cell'deki arıza o cell'le sınırlı kalır, diğer cell'lerdeki tenant'lar etkilenmez.

#### 4. Per-Tenant Backup and Restore
Her tenant'ın shard'ını ayrı ayrı yedeklemek, bir tenant'ın durumunu diğer tenant'ları etkilemeden yedekten geri yüklemeyi mümkün kılar. Bu özellikle tenant'ın önemli verileri yanlışlıkla silmesi veya üzerine yazması durumunda yararlıdır.

#### 5. Regulatory Compliance (Düzenleyici Uyumluluk)
**GDPR** ve **CCPA** gibi veri gizliliği düzenlemeleri, bireylere işletmelerin hakkında sakladığı kişisel bilgilere erişme ve bu bilgilerin silinmesini talep etme hakkı tanır. Her kişinin verisi ayrı bir shard'da saklanıyorsa, bu durum shard üzerinde basit veri dışa aktarma ve silme işlemlerine dönüşür.

#### 6. Data Residence (Veri İkamet Zorunluluğu)
Belirli bir tenant'ın verisinin **data residency** (veri ikameti) yasalarına uymak için belirli bir yargı bölgesinde saklanması gerekiyorsa, **region-aware** (bölge bilinçli) bir veritabanı, o tenant'ın shard'ını belirli bir bölgeye atamanıza olanak tanır.

#### 7. Gradual Schema Rollout (Kademeli Şema Güncelleme)
**Schema migration**'lar (şema geçişleri) tek bir seferde tüm tenant'lara uygulanmak yerine, bir tenant'tan diğerine kademeli olarak yayılabilir. Bu risk azaltır; sorunları tüm tenant'ları etkilemeden önce fark edebilirsiniz. Ancak bu işlemi transaction'al olarak yapmak zordur.

### Multitenancy'de Sharding'in Zorlukları

1. **Her tenant tek bir node'a sığmalıdır.** Tek bir node'a sığmayacak kadar büyük bir tenant varsa, o tenant içinde de sharding uygulamak gerekir; bu da bizi scalability için sharding konusuna geri götürür.

2. **Çok sayıda küçük tenant için ayrı shard yaratmak çok fazla ek yük getirebilir.** Birkaç küçük tenant'ı aynı shard'da gruplamak bir çözüm olsa da, büyüdükçe tenant'ları bir shard'dan diğerine taşıma sorunuyla karşılaşılır.

3. **Birden fazla tenant'ın verilerini birleştiren özellikler uygulamak zorlaşır.** Verilerin birden fazla shard'a dağıldığı durumlarda cross-shard join gerektirir.

## 🔑 Key-Value Verinin Sharding'i

### Temel Hedef: Dengeli Yük Dağılımı

Sharding'in amacı, veriyi ve sorgu yükünü node'lar arasında **eşit** dağıtmaktır. Her node adil bir pay alırsa, teorik olarak 10 node; tek bir node'un 10 katı veriyi ve okuma/yazma kapasitesini kaldırabilir (replication göz ardı edildiğinde).

Node eklediğinizde veya kaldırdığınızda, yeni node sayısına eşit olarak dağıtılmış olmasını sağlamak için yükü **rebalance** (yeniden dengelemek) etmek de isteyebilirsiniz.

### Skew ve Hot Spot Nedir?

Sharding adaletsizse, bazı shard'lar diğerlerinden daha fazla veri veya sorgu içeriyorsa, buna **skewed** (eğik/dengesiz) denir. Aşırı bir durumda tüm yük tek bir shard'a düşebilir; 10 node'un 9'u boşta kalırken tüm iş tek meşgul node'dan geçer:

- Orantısız biçimde yüksek yük taşıyan shard'a **hot shard** veya **hot spot** denir.
- Özellikle yüksek yük altında olan tek bir key varsa (örneğin sosyal ağdaki bir ünlü kullanıcı) buna **hot key** denir.

### Partition Key ve Sharding Algoritması

Veri kümesini shard'lara bölmek için bir **partition key** alıp hangi shard'ın o record'u içerdiğini söyleyen bir algoritmaya ihtiyaç duyulur:
- Bir **key-value store**'da partition key genellikle key'in kendisi veya ilk kısmıdır.
- **Relational model**de partition key, bir tablonun sütunu olabilir (mutlaka birincil anahtar olmak zorunda değildir).

Bu algoritmanın hot spot'ları gidermek için **rebalancing**'e uygun olması gerekir.

## 📏 Key Range Sharding (Anahtar Aralığına Göre Sharding)

### Temel Fikir

Her shard'a belirli bir **partition key** aralığı (minimumdan maksimuma) atanır; tıpkı bir kâğıt ansiklopedisinin ciltleri gibi. Örneğin:
- Cilt 1: A–B harfleriyle başlayan maddeler
- Cilt 12: T, U, V, W, X, Y, Z harfleriyle başlayan maddeler

Bir kaydı aramak için hangi cilt/shard'da olduğunu belirlemek ve doğru shard'ı seçmek yeterlidir.

### Shard Sınırları Eşit Aralıklı Olmak Zorunda Değildir

Veriniz eşit dağılmamış olabilir. Her iki harf için bir cilt olması bazı ciltlerin diğerlerinden çok daha büyük olmasına yol açar. Veriyi eşit dağıtmak için **shard boundary**'lerin (shard sınırlarının) veriye adapte olması gerekir.

### Shard Sınırlarını Kim Belirler?

- **Manuel:** Bir yönetici tarafından elle belirlenir. Örneğin **Vitess** (MySQL için sharding katmanı) manual key-range sharding kullanır.
- **Otomatik:** Veritabanı tarafından otomatik seçilir. **Bigtable** ve açık kaynak muadili **HBase**, **MongoDB**'nin range tabanlı sharding seçeneği, **CockroachDB**, **RethinkDB**, **FoundationDB** otomatik variant kullanır. **YugabyteDB** hem manuel hem otomatik tablet bölmeyi destekler.

### Avantaj: Sorted Order ve Range Queries

Her shard içinde key'ler sıralı düzende saklanır (örneğin **B-tree** veya **SSTables**'da). Bu sayede **range scan**'ler (aralık taramaları) kolaydır ve birden fazla ilgili record'u tek bir sorguda getirmek için key bir **concatenated index** (birleşik index) olarak kullanılabilir.

**Örnek:** Bir sensör ağından veri saklayan ve key olarak ölçüm timestamp'ini kullanan bir uygulama düşünün. Range scan'ler bu durumda çok faydalıdır: belirli bir aya ait tüm okumaları kolayca çekebilirsiniz.

### Dezavantaj: Hot Shard Riski

Key-range sharding ile, bitişik key'lere çok sayıda yazma işlemi varsa kolayca **hot shard** oluşabilir. Örneğin key bir timestamp ise shard'lar zaman aralıklarına karşılık gelir (bir shard = bir ay). Sensörlerden veri geldiği an veritabanına yazıyorsanız, tüm yazma işlemleri aynı shard'a (bu aydaki) gider; bu shard yazma yükünün altında ezilirken diğerleri boşta kalır.

### Hot Shard Sorununa Çözüm

Bu sorunu çözmek için key'in ilk elemanı olarak timestamp yerine başka bir şey kullanmak gerekir. Örneğin, her timestamp'in önüne **sensor ID** eklenebilir; bu sayede key sıralaması önce sensor ID'ye, sonra timestamp'e göre yapılır. Aynı anda aktif birçok sensör varsa yazma yükü shard'lara daha dengeli dağılır. Ancak bir zaman aralığında birden fazla sensörün değerlerini almak istediğinizde, artık her sensör için ayrı bir range sorgusu yapmak gerekir.

### Key-Range Sharding'de Rebalancing

#### Pre-splitting
Veritabanını ilk kurduğunuzda bölünecek key range'leri yoktur. **HBase** ve **MongoDB** gibi bazı veritabanları boş bir veritabanında başlangıç shard kümesi yapılandırmanıza izin verir; buna **pre-splitting** denir. Bu, key dağılımının nasıl görüneceğine dair önceden bir fikrinizin olmasını gerektirir.

#### Shard Splitting
Veri hacmi ve write throughput arttıkça, key-range sharding kullanan bir sistem büyür: mevcut bir shard'ı iki ya da daha fazla küçük shard'a **split** (böler); her biri orijinal key range'inin bitişik bir alt aralığını barındırır. Oluşan küçük shard'lar ardından birden fazla node'a dağıtılabilir.

Büyük miktarda veri silinirse, küçülmüş birkaç bitişik shard'ı tek bir büyük shard'da **merge** (birleştirmek) etmek de gerekebilir.

Bu süreç, **B-tree**'nin üst seviyesinde olanlarla benzerlik taşır.

#### Split Tetikleyicileri
Shard sınırlarını otomatik yöneten veritabanlarında, shard split genellikle şu koşullarda tetiklenir:
- Shard yapılandırılmış boyuta ulaştığında (HBase'de varsayılan 10 GB)
- Bazı sistemlerde write throughput belirli bir eşiğin kalıcı olarak üzerine çıktığında

Böylece çok fazla veri saklamıyor olsa bile, write yükü daha dengeli dağıtılabilmesi için hot bir shard bölünebilir.

Shard sayısı veri hacmine uyum sağlar: az veri varsa az shard yeterlidir (yük azalır); büyük miktarda veri varsa her shard'ın boyutu yapılandırılabilir maksimumla sınırlıdır.

#### Shard Split'in Maliyeti
Shard split pahalı bir işlemdir; tüm verinin yeni dosyalara yeniden yazılmasını gerektirir. Bu, **log-structured storage engine**'deki compaction işlemine benzer. Split gerektiren shard genellikle yüksek yük altındadır; split maliyeti bu yükü daha da artırabilir.

In [ ]:
# Key-Range Sharding Örneği
# Timestamp tabanlı sensor verisini shard'lara bölme

import hashlib
from datetime import datetime, timedelta

# --- YANLIŞ YAKLAŞIM: Sadece timestamp key ---
def wrong_key(sensor_id, timestamp):
    return f"{timestamp}_{sensor_id}"

# --- DOĞRU YAKLAŞIM: sensor_id + timestamp bileşik key ---
def correct_key(sensor_id, timestamp):
    return f"{sensor_id}_{timestamp}"

# Örnek veriler
sensors = ["sensor_001", "sensor_002", "sensor_003"]
now = datetime(2024, 1, 1, 10, 0, 0)

print("=== YANLIŞ: Sadece timestamp key ===")
print("Tüm yazma işlemleri aynı shard'a (o anki) gider!")
for i in range(3):
    ts = (now + timedelta(seconds=i)).strftime("%Y%m%d%H%M%S")
    for sensor in sensors:
        key = wrong_key(sensor, ts)
        print(f"  Key: {key}")

print("\n=== DOĞRU: sensor_id + timestamp key ===")
print("Yazma işlemleri farklı shard'lara dağılır!")
for i in range(3):
    ts = (now + timedelta(seconds=i)).strftime("%Y%m%d%H%M%S")
    for sensor in sensors:
        key = correct_key(sensor, ts)
        print(f"  Key: {key}")

## 🎲 Hash of Key ile Sharding (Hash Tabanlı Sharding)

### Temel Fikir

Key-range sharding, bitişik (ancak farklı) partition key'lere sahip record'ların aynı shard'da gruplanmasını istediğinizde faydalıdır (örneğin timestamp'lerle). Partition key'lerin birbirine yakın olup olmamasının önemi yoksa (örneğin multitenant uygulamada tenant ID'leri), yaygın bir yaklaşım partition key'i bir shard'a eşlemeden önce **hash**'lemektir.

### İyi Bir Hash Fonksiyonu Ne Yapar?

İyi bir **hash function**, dengesiz (skewed) veriyi alıp eşit dağıtır. 32-bit bir hash fonksiyonu düşünün: bir string verildiğinde 0 ile 2³²−1 arasında görünüşte rastgele bir sayı döndürür. Girdi string'leri ne kadar benzer olursa olsun, hash'leri bu aralık üzerinde eşit dağılır (ancak aynı girdi her zaman aynı çıktıyı verir).

### Sharding için Hangi Hash Fonksiyonu?

Sharding amaçlı kullanım için hash fonksiyonu kriptografik açıdan güçlü olmak zorunda değildir:
- **MongoDB** → MD5 kullanır
- **Cassandra ve ScyllaDB** → Murmur3 kullanır

Pek çok programlama dilinde yerleşik hash fonksiyonları bulunur, ancak bunlar sharding için uygun olmayabilir:
- **Java**'nın `Object.hashCode()` ve **Ruby**'nin `Object#hash` metotlarında aynı key farklı process'lerde farklı hash değeri üretebilir; bu, sharding için kullanılamaz.

### Hash Modulo Sayısı ile Shard Seçimi (Neden Kötü?)

Key hash'lendikten sonra hangi shard'da saklanacağını seçmenin ilk akla gelen yöntemi, hash değerini sistemdeki node sayısına bölüp kalanını almaktır (**mod N** yaklaşımı). Örneğin `hash(key) % 10`, 0 ile 9 arasında bir sayı döndürür; 10 node varsa bu yeterli görünür.

**Sorun:** Node sayısı N değişirse, key'lerin büyük çoğunluğu bir node'dan diğerine taşınmak zorunda kalır. Örneğin:
- 3 node varken `hash=3` → node 0'da (3 % 3 = 0)
- 4 node eklendiğinde `hash=3` → node 3'e taşınır (3 % 4 = 3)

Mod N fonksiyonu hesaplaması kolaydır ancak çok verimsiz **rebalancing**'e yol açar; record'ların gereksiz yere bir node'dan diğerine taşınması söz konusudur. Olabildiğince az veri taşınan bir yaklaşıma ihtiyaç duyulur.

In [ ]:
# Mod N yaklaşımının problemi: node sayısı değiştiğinde fazla veri taşınır

def get_shard_mod_n(key_hash, num_nodes):
    return key_hash % num_nodes

# 3 node ile başla
num_nodes_before = 3
num_nodes_after = 4

# 12 key için hash değerleri simüle et
key_hashes = list(range(12))

print(f"Node sayısı: {num_nodes_before} → {num_nodes_after}")
print(f"{'Key Hash':<12} {'Eski Shard':<15} {'Yeni Shard':<15} {'Taşındı?':<10}")
print("-" * 55)

moved = 0
total = len(key_hashes)

for h in key_hashes:
    old_shard = get_shard_mod_n(h, num_nodes_before)
    new_shard = get_shard_mod_n(h, num_nodes_after)
    moved_flag = "✅ EVET" if old_shard != new_shard else "❌ HAYIR"
    if old_shard != new_shard:
        moved += 1
    print(f"{h:<12} {old_shard:<15} {new_shard:<15} {moved_flag}")

print(f"\nToplam {total} key'den {moved} tanesi taşındı ({moved/total*100:.0f}%)")
print("\nSonuç: Mod N yöntemi ile node sayısı değiştiğinde büyük çoğunluk taşınır!")

## 🔢 Fixed Number of Shards (Sabit Shard Sayısı)

### Temel Fikir

Basit ama yaygın olarak kullanılan bir çözüm: node sayısından çok daha fazla shard oluşturmak ve her node'a birden fazla shard atamak. Örneğin:
- 10 node'dan oluşan bir cluster'da 1.000 shard oluşturulur
- Her node'a 100 shard atanır
- Bir key, `hash(key) % 1.000` numaralı shard'da saklanır
- Sistem, hangi shard'ın hangi node'da olduğunu ayrıca izler

### Rebalancing Nasıl Çalışır?

Cluster'a yeni bir node eklendiğinde, sistem mevcut node'lardan bazı shard'ları yeni node'a yeniden atar; dağılım tekrar adil hale gelene kadar devam eder. Bir node kaldırıldığında tam tersi gerçekleşir.

Bu modelde:
- Yalnızca **tüm shard'lar** node'lar arasında taşınır (shard'ları bölmek yerine)
- Shard sayısı değişmez
- Key'lerin shard'lara atanması değişmez
- Yalnızca **shard'ların node'lara atanması** değişir
- Bu yeniden atama anlık değildir; büyük miktarda verinin ağ üzerinden aktarılması zaman alır; bu süre boyunca okuma/yazma işlemleri eski atamayı kullanır.

### Shard Sayısını Seçme

Shard sayısının çok sayıda çarpanla bölünebilir bir sayı olması yaygın bir yaklaşımdır; böylece dataset çeşitli node sayılarına eşit biçimde dağıtılabilir. Bu sayede node sayısının 2'nin kuvveti olma zorunluluğu ortadan kalkar. Cluster'daki donanım farklılıklarına bile uyum sağlanabilir: daha güçlü node'lara daha fazla shard atanarak bu node'ların yükün daha büyük bir bölümünü üstlenmesi sağlanır.

### Hangi Sistemler Kullanıyor?

Bu sharding yaklaşımı **Citus** (PostgreSQL için sharding katmanı), **Riak**, **Elasticsearch** ve **Couchbase** tarafından kullanılmaktadır. Başlangıçta kaç shard gerektiğini iyi tahmin edebildiğiniz sürece iyi çalışır; sahip olduğunuz shard sayısından daha fazla node edinemeyeceğiniz sınırına dikkat etmek kaydıyla node'ları kolayca ekleyip kaldırabilirsiniz.

### Başlangıç Shard Sayısının Yanlış Olması

Başlangıçta yapılandırılan shard sayısının yanlış olduğunu fark ederseniz (örneğin shard sayısından daha fazla node'a ihtiyaç duyacak ölçeğe ulaşırsanız), pahalı bir **resharding** işlemi gerekir: her shard'ı bölüp yeni dosyalara yazmak, süreçte fazladan disk alanı kullanmak anlamına gelir. Bazı sistemler veri tabanına aynı anda yazma yapılırken resharding'e izin vermez; bu da shard sayısını kesinti yaşamadan değiştirmeyi güçleştirir.

### Doğru Shard Sayısını Seçmenin Güçlüğü

Dataset boyutu çok değişkense (örneğin başlangıçta küçük ama zamanla çok büyüyebiliyorsa) doğru shard sayısını seçmek zordur. Her shard toplam verinin sabit bir bölümünü içerdiğinden, her shard'ın boyutu cluster'daki toplam veri miktarıyla orantılı büyür. Shard'lar çok büyükse rebalancing ve node arızalarından kurtarma pahalı olur. Çok küçükse aşırı ek yük (overhead) oluşur. En iyi performans, shard boyutunun "tam doğru" olduğunda elde edilir; ancak shard sayısı sabitken dataset boyutu değiştiğinden bunu başarmak zordur.

## 🌊 Hash Range Sharding (Hash Aralığına Göre Sharding)

### Temel Fikir

Gerekli shard sayısı önceden tahmin edilemiyorsa, shard sayısının iş yüküne kolayca adapte olabildiği bir yaklaşım kullanmak daha iyidir. Key-range sharding bu özelliğe sahiptir, ancak bitişik key'lere çok sayıda yazma işlemi olduğunda hot spot riski taşır.

Bir çözüm, key-range sharding ile hash fonksiyonunu birleştirmektir: her shard, key aralığı yerine **hash değerleri aralığı** içerir.

**Örnek:** 0 ile 65.535 (2¹⁶ − 1) arasında sayı döndüren 16-bit bir hash fonksiyonu kullanalım. Girdi key'leri ne kadar benzer olursa olsun (örneğin ardışık timestamp'ler), hash'leri bu aralığa eşit dağılır. Her shard'a bir hash değeri aralığı atanabilir:
- Shard 0: 0 → 16.383
- Shard 1: 16.384 → 32.767
- ... vb.

### Key-Range Sharding ile Karşılaştırma

Hash-range sharding'de de, key-range sharding'de olduğu gibi, bir shard çok büyük ya da çok yüklü olduğunda bölünebilir. Bu hâlâ pahalı bir işlemdir, ancak gerektiğinde yapılabilir; böylece shard sayısı önceden sabitlenmek yerine veri hacmine göre adapte olur.

**Dezavantajı:** Key-range sharding'e kıyasla partition key üzerindeki range sorguları verimli değildir; aralıktaki key'ler tüm shard'lara dağılmış durumdadır.

**Ancak**, key'ler iki veya daha fazla sütundan oluşuyorsa ve partition key bunların yalnızca ilkiyse, ikinci ve sonraki sütunlar üzerinde verimli range sorguları yapılabilir; aynı partition key'e sahip tüm record'lar aynı shard'da olduğundan.

### Veri Ambarlarında Sharding

**BigQuery**, **Snowflake** ve **Delta Lake** gibi veri ambarları benzer bir indexleme yaklaşımını destekler (terminoloji farklı olsa da):
- **BigQuery'de:** Partition key, bir record'un hangi partition'da bulunduğunu belirler; "cluster column"lar, partition içindeki record'ların nasıl sıralandığını belirler.
- **Snowflake:** Record'ları otomatik olarak "micro-partition"lara atar, ancak kullanıcıların bir tablo için cluster key tanımlamasına izin verir.
- **Delta Lake:** Hem manuel hem otomatik partition atamasını ve cluster key'leri destekler.

Veriyi kümelemek yalnızca range scan performansını değil, sıkıştırma ve filtreleme performansını da iyileştirebilir.

### Hash-Range Kullanan Sistemler

**YugabyteDB** ve **DynamoDB** hash-range sharding kullanır; **MongoDB**'de de bir seçenek olarak mevcuttur. **Cassandra** ve **ScyllaDB** bu yaklaşımın özel bir varyantını kullanır:

- Olası hash değerleri alanı, node sayısıyla orantılı sayıda aralığa bölünür (Cassandra'da varsayılan olarak node başına 16, ScyllaDB'de node başına 256 aralık), rastgele sınırlarla.
- Bazı aralıklar diğerlerinden büyük olsa da, node başına birden fazla aralık olduğu için bu dengesizlikler birbirini dengeler.
- Node eklendikçe veya çıkarıldıkça, aralık sınırları ayarlanır ve shard'lar bölünüp birleştirilir.
- Yeni bir node eklendiğinde bazı aralıklar mevcut node'lardan yeni node'a aktarılır; bu yeni node'a yaklaşık olarak adil bir pay verir, node'lar arasında gerektiğinden fazla veri aktarmadan.

## 🔄 Consistent Hashing (Tutarlı Hashleme)

### Tanım

Bir **consistent hashing** algoritması, key'leri belirli sayıda shard'a şu iki özelliği sağlayacak biçimde eşleyen bir hash fonksiyonudur:

1. Her shard'a eşlenen key sayısı yaklaşık olarak eşittir.
2. Shard sayısı değiştiğinde, mümkün olduğunca az key bir shard'dan diğerine taşınır.

> **Not:** Buradaki *consistent* (tutarlı), **replica consistency** (Bölüm 6) veya **ACID consistency** (Bölüm 8) ile ilgisi yoktur; daha çok bir key'in mümkünse aynı shard'da kalma eğilimini ifade eder.

### Cassandra ve ScyllaDB'nin Algoritması

**Cassandra** ve **ScyllaDB**'nin kullandığı sharding algoritması, consistent hashing'in orijinal tanımına benzer. Bunun yanı sıra başka consistent hashing algoritmaları da önerilmiştir:
- **Highest random weight** (aynı zamanda **rendezvous hashing** olarak da bilinir)
- **Jump consistent hashing**

Bu yaklaşımlarla, eklenen node için mevcut az sayıda shard'ı alt aralıklara bölerek yeni shard'lar oluşturmak yerine, yeni node'a daha önce diğer tüm node'lara dağılmış bireysel key'ler atanır. Hangisinin tercih edileceği uygulamaya bağlıdır.

In [ ]:
# Consistent Hashing Basit Örnek - Rendezvous Hashing
import hashlib

def rendezvous_hash(key, nodes):
    """Rendezvous hashing: en yüksek hash değerini veren node'a ata"""
    best_node = None
    best_score = -1
    for node in nodes:
        score = int(hashlib.md5(f"{key}:{node}".encode()).hexdigest(), 16)
        if score > best_score:
            best_score = score
            best_node = node
    return best_node

# 3 node ile başla
nodes_before = ["node_0", "node_1", "node_2"]
nodes_after = ["node_0", "node_1", "node_2", "node_3"]  # node_3 eklendi

keys = [f"key_{i}" for i in range(20)]

print("Rendezvous Hashing - Node Ekleme Etkisi")
print(f"{'Key':<12} {'Eski Node':<15} {'Yeni Node':<15} {'Taşındı?':<10}")
print("-" * 55)

moved = 0
for key in keys:
    old = rendezvous_hash(key, nodes_before)
    new = rendezvous_hash(key, nodes_after)
    moved_flag = "✅ EVET" if old != new else "❌ HAYIR"
    if old != new:
        moved += 1
    print(f"{key:<12} {old:<15} {new:<15} {moved_flag}")

print(f"\n{len(keys)} key'den {moved} tanesi taşındı ({moved/len(keys)*100:.0f}%)")
print("Consistent hashing ile minimum veri taşınır!")

## 🔥 Skewed Workloads ve Hot Spot Giderme

### Sorun: Uniform Dağılım Uniform Yük Anlamına Gelmez

Consistent hashing, key'lerin node'lara eşit dağıtıldığını garanti eder; ancak bu, gerçek yükün eşit dağılacağı anlamına gelmez. İş yükü yüksek oranda **skewed** (dengesiz) ise, yani bazı partition key'lerin altında diğerlerinden çok daha fazla veri bulunuyorsa ya da bazı key'lere yapılan istek oranı çok yüksekse, bazı sunucular aşırı yüklenirken diğerleri neredeyse boşta kalabilir.

### Gerçek Hayat Örneği: Ünlü Etkisi

Bir sosyal medya sitesinde milyonlarca takipçisi olan bir ünlü kullanıcının paylaşımı yoğun bir aktivite fırtınasına yol açabilir. Bu, aynı key'e (partition key belki kullanıcının ID'si ya da insanların yorum yaptığı eylemin ID'si) büyük hacimde okuma ve yazma yapılmasıyla sonuçlanabilir.

### Çözüm 1: Esnek Sharding Politikası

Bu tür durumlarda daha esnek bir sharding politikası gereklidir. Key aralıklarını (veya hash aralıklarını) baz alan bir sistem, bireysel bir hot key'i kendi başına bir shard'a yerleştirmeyi ve hatta ona özel bir makine atamayı mümkün kılar.

### Çözüm 2: Uygulama Katmanında Skew Giderme

**Uygulama düzeyinde** de skew giderilebilir. Örneğin, çok sıcak (hot) olduğu bilinen bir key varsa, key'in başına veya sonuna rastgele bir sayı eklemek işe yarar. Yalnızca iki rastgele rakam eklemek, o key'e yapılan yazma işlemlerini 100 key'e eşit biçimde dağıtır; bu key'ler farklı shard'lara dağıtılabilir.

**Dezavantajı:** Yazma işlemleri birden fazla key'e bölündüğünde, okuma işlemlerinin ek iş yapması gerekir; 100 key'in tamamından veri okunup birleştirilmek zorunda kalınır. Hot key'in her shard'ındaki okuma yükü azalmaz; yalnızca yazma yükü bölünür.

Bu teknik ek **bookkeeping** (kayıt tutma) gerektirir: rastgele sayı yalnızca az sayıda hot key için eklenmeli; yazma throughput'u düşük olan büyük çoğunluktaki key'ler için bu gereksiz ek yük oluşturur. Hangi key'lerin bölündüğünü takip etmek ve normal bir key'i özel yönetilen hot key'e dönüştürmek için bir süreç gerekmektedir.

### Zaman İçinde Değişen Yük

Sorun, yükün zamanla değişmesiyle daha da karmaşıklaşır. Örneğin viral hale gelen bir sosyal medya gönderisi birkaç gün boyunca yüksek yük alabilir, ardından büyük olasılıkla sakinleşir. Bunun yanı sıra bazı key'ler yazma için, bazıları okuma için sıcak olabilir; bu da farklı stratejiler gerektiren farklı durumlar doğurur.

### Otomatik Çözümler

Büyük ölçek için tasarlanmış bazı sistemler (özellikle bulut hizmetleri) hot shard'larla başa çıkmak için otomatik yaklaşımlar geliştirmiştir. **Amazon** bunu **heat management** (ısı yönetimi) veya **adaptive capacity** (uyarlanabilir kapasite) olarak adlandırmaktadır.

In [ ]:
# Hot Key Giderme: Rastgele Suffix Tekniği
import random
import hashlib
from collections import Counter

def get_shard(key, num_shards):
    h = int(hashlib.md5(key.encode()).hexdigest(), 16)
    return h % num_shards

NUM_SHARDS = 5
NUM_WRITES = 1000
NUM_SUFFIXES = 10  # Kaç farklı suffix kullanılacak

# --- Senaryo 1: Hot Key - tüm yazma işlemleri aynı key'e gider ---
hot_key = "celebrity_user_123"
shard_counter_hot = Counter()
for _ in range(NUM_WRITES):
    shard = get_shard(hot_key, NUM_SHARDS)
    shard_counter_hot[shard] += 1

# --- Senaryo 2: Hot Key + Random Suffix ---
shard_counter_distributed = Counter()
for _ in range(NUM_WRITES):
    suffix = random.randint(0, NUM_SUFFIXES - 1)
    modified_key = f"{hot_key}_{suffix}"
    shard = get_shard(modified_key, NUM_SHARDS)
    shard_counter_distributed[shard] += 1

print("=== Senaryo 1: Hot Key (Random Suffix YOK) ===")
print(f"Tüm {NUM_WRITES} yazma işlemi aynı shard'a gider:")
for shard_id in range(NUM_SHARDS):
    count = shard_counter_hot.get(shard_id, 0)
    bar = "█" * (count // 20)
    print(f"  Shard {shard_id}: {count:4d} yazma {bar}")

print(f"\n=== Senaryo 2: Hot Key + {NUM_SUFFIXES} Random Suffix ===")
print("Yazma işlemleri farklı shard'lara dağılır:")
for shard_id in range(NUM_SHARDS):
    count = shard_counter_distributed.get(shard_id, 0)
    bar = "█" * (count // 20)
    print(f"  Shard {shard_id}: {count:4d} yazma {bar}")

print("\nNOT: Okuma yapılırken tüm suffix'li key'lerin birleştirilmesi gerekir!")

## ⚙️ Otomatik vs. Manuel Rebalancing

### Temel Soru

Shard'ların bölünmesi ve rebalancing'i otomatik mi yoksa manuel mi yapılmalıdır?

### Üç Yaklaşım

1. **Tam Otomatik:** Sistem herhangi bir insan müdahalesi olmadan shard'ları ne zaman bölüp hangi node'a taşıyacağına kendisi karar verir.
2. **Tam Manuel:** Sharding'i açıkça yönetmesi için bir yönetici atanır.
3. **Karma (Middle Ground):** Örneğin **Couchbase** ve **Riak**, önerilen shard atamasını otomatik oluşturur ancak yürürlüğe girmeden önce bir yöneticinin onayını gerektirir.

### Tam Otomatik Rebalancing'in Avantajları

- Normal bakım için daha az operasyonel iş
- Sistemler, iş yükündeki değişikliklere adapte olmak için **autoscale** yapabilir
- **DynamoDB** gibi bulut veritabanları, artan/azalan yüke birkaç dakika içinde uyum sağlamak için otomatik shard ekleme/kaldırma yapabildiğini iddia eder

### Tam Otomatik Rebalancing'in Riskleri

- Rebalancing pahalı bir işlemdir: istekleri yeniden yönlendirmeyi ve büyük miktarda veriyi bir node'dan diğerine taşımayı gerektirir. Dikkatli yapılmazsa ağı veya node'ları aşırı yükleyebilir ve diğer isteklerin performansına zarar verebilir.
- Sistem, rebalancing devam ederken yazma işlemlerini işlemeye devam etmek zorundadır; bir sistem maksimum write throughput'una yakınsa, shard-splitting süreci gelen yazmaların hızına yetişemeyebilir.

### Otomasyonun Tehlikeli Kombinasyonu

Otomatik rebalancing, otomatik arıza tespiti ile birleşince tehlikeli olabilir:

Aşırı yüklenmiş bir node geçici olarak yavaş yanıt vermeye başlar → Diğer node'lar bu node'un öldüğüne karar verir → Yükü bu node'dan uzaklaştırmak için otomatik rebalancing başlar → Diğer node'lara ve ağa ek yük bindirir → Durum kötüleşir → Diğer node'lar da aşırı yüklenir ve hatalı biçimde kapalı zannedilir.

Bu bir **cascading failure** (kademeli arıza) riskidir.

### Manuel Rebalancing'in Avantajları

Bu nedenle rebalancing sürecine bir insanın dahil olması iyi olabilir. Tam otomatik süreçten daha yavaş olsa da, operasyonel sürprizleri önlemeye yardımcı olabilir. Manuel rebalancing aynı zamanda şu durumlarda da faydalıdır:
- Bilinen bir etkinlik nedeniyle trafik artışı bekleniyor (örneğin Cyber Monday tatil indirimleri ya da Dünya Kupası gibi popüler etkinliklerin bilet satışları)

## 🗺️ Request Routing (İstek Yönlendirme)

### Sorun

Bir dataset'i birden fazla node'a shard'ladık ve shard'ları node'lar değiştikçe rebalance ettik. Şimdi başka bir soru var: belirli bir key okumak veya yazmak istediğinizde, hangi node'a (yani hangi IP adresi ve port numarasına) bağlanmanız gerektiğini nasıl bileceksiniz?

Bu soruna **request routing** denir ve daha önce "Load balancers, service discovery ve service meshes" bölümünde tartışılan **service discovery** ile çok benzerdir.

**Temel fark:** Uygulama kodunu çalıştıran servislerle (genellikle stateless olan) bir load balancer isteği herhangi bir instance'a gönderebilir. Sharded veritabanlarında ise bir key'e yönelik istek, yalnızca o key'i içeren shard'ın replikası olan bir node tarafından karşılanabilir.

### Üç Request Routing Yaklaşımı

Request routing, key'leri shard'lara ve shard'ları node'lara eşleyen bir eşleşme hakkında bilgi sahibi olmak zorundadır. Üç yüksek düzeyli yaklaşım vardır:

#### Yaklaşım 1: İstemci Herhangi Bir Node'a Bağlanabilir
İstemciler herhangi bir node'a bağlanabilir (örneğin **round-robin load balancer** aracılığıyla). O node, istek ilgili shard'ı barındırıyorsa doğrudan işler; değilse uygun node'a iletir, yanıtı alır ve istemciye iletir.

#### Yaklaşım 2: Routing Tier
Tüm istekler önce bir **routing tier**'e (yönlendirme katmanına) gönderilir. Bu katman her isteği işlemesi gereken node'u belirleyip uygun şekilde yönlendirir. Routing tier, istekleri kendisi işlemez; yalnızca **shard-aware load balancer** olarak görev yapar.

#### Yaklaşım 3: Akıllı İstemci
İstemcilerin sharding ve shard'ların node'lara atanması hakkında bilgi sahibi olması gerekir. Bu durumda istemci, herhangi bir aracı olmadan doğrudan uygun node'a bağlanabilir.

### Request Routing'deki Temel Sorunlar

Her durumda çeşitli kritik sorunlar vardır:

**1. Koordinatör Seçimi:** Hangi shard'ın hangi node'da olduğuna kim karar verir? En basiti tek bir koordinatörün bu kararı vermesidir, ancak o zaman koordinatör çalışan node kapalıya gittiğinde buna nasıl **fault-tolerant** olunur? Koordinatör rolü başka bir node'a devredebiliyorsa, iki farklı koordinatörün çelişkili shard atamaları yaptığı **split-brain** durumunu nasıl engellersiniz?

**2. Yönlendirme Bileşeni Nasıl Güncellenir?** Yönlendirmeyi gerçekleştiren bileşen (node'lardan biri, routing tier ya da istemci), shard'ların node'lara atanmasındaki değişikliklerden nasıl haberdar olur?

**3. Cutover Dönemi:** Bir shard bir node'dan diğerine taşınırken, yeni node devralır ancak eski node'a yapılan istekler hâlâ gelmeye devam edebilir. Bunlar nasıl ele alınır?

### ZooKeeper ve etcd ile Koordinasyon

Pek çok distributed veri sistemi, shard atamalarını takip etmek için **ZooKeeper** veya **etcd** gibi ayrı bir koordinasyon servisi kullanır. **Consensus algoritmaları** kullanarak fault tolerance ve split brain koruması sağlarlar.

Her node kendini ZooKeeper'a kaydeder; ZooKeeper, shard'ların node'lara yetkili eşleşmesini tutar. Routing tier veya shard-aware istemci gibi diğer aktörler bu bilgiye ZooKeeper'dan abone olabilir. Bir shard sahipliği değiştiğinde ya da bir node eklenip kaldırıldığında ZooKeeper, routing tier'i bildirir; böylece yönlendirme bilgisi güncel tutulur.

### Farklı Sistemlerin Yaklaşımları

| Sistem | Koordinasyon Yaklaşımı |
|--------|------------------------|
| **HBase**, **SolrCloud** | ZooKeeper kullanır |
| **Kubernetes** | etcd ile hangi servis instance'ının nerede çalıştığını takip eder |
| **MongoDB** | Kendi config server implementasyonu ve mongos daemon'larını routing tier olarak kullanır |
| **Kafka**, **YugabyteDB**, **TiDB**, **ScyllaDB** | Bu koordinasyon için yerleşik Raft consensus protokolü implementasyonları kullanır |
| **Riak** | Farklı bir yaklaşım: cluster durumundaki değişiklikleri yaymak için node'lar arasında **gossip protocol** kullanır (çok daha zayıf tutarlılık sağlar) |

### Gossip Protocol'un Zayıf Tutarlılığı

Riak'ın gossip protocol yaklaşımı, consensus protocol'den çok daha zayıf tutarlılık sağlar; cluster'ın farklı bölümlerinin aynı shard için farklı node atamalarına sahip olduğu **split brain** durumu mümkündür. **Leaderless** veritabanları bunu tolere edebilir; zaten genellikle zayıf tutarlılık garantileri verirler.

### IP Adresleri için DNS

Bir routing tier kullanılırken veya rastgele bir node'a istek gönderilirken istemcilerin bağlanacakları IP adreslerini bulmaya ihtiyacı vardır. Bunlar, shard'ların node'lara atanması kadar hızlı değişmediğinden, bu amaçla genellikle **DNS** kullanmak yeterlidir.

### Analitik Veritabanlarında Request Routing

Request routing tartışması ağırlıklı olarak bireysel key'i bulmakla ilgilidir; bu, sharded OLTP veritabanları için en önemli konudur. Analitik veritabanları da sharding kullanır, ancak çok farklı türde sorgu yürütme gerektirir: tek bir shard'da çalışmak yerine, bir sorgunun genellikle birçok shard'dan paralel olarak veriyi toplaması ve birleştirmesi gerekir. Bu tür **paralel sorgu yürütme** teknikleri Bölüm 11'de ele alınacaktır.

In [ ]:
# Request Routing: 3 Yaklaşımı Simüle Eden Sınıflar

import hashlib

class ShardRouter:
    """Shard routing mantığını simüle eder"""
    
    def __init__(self, num_shards=8):
        self.num_shards = num_shards
        # Shard → Node eşleşmesi (ZooKeeper bu bilgiyi tutar)
        self.shard_to_node = {i: f"node_{i % 4}" for i in range(num_shards)}
        self.request_log = []
    
    def get_shard(self, key):
        h = int(hashlib.md5(key.encode()).hexdigest(), 16)
        return h % self.num_shards
    
    def get_node(self, key):
        return self.shard_to_node[self.get_shard(key)]
    
    # --- Yaklaşım 1: Routing Tier ---
    def routing_tier_approach(self, key):
        shard = self.get_shard(key)
        target_node = self.shard_to_node[shard]
        log = f"[Routing Tier] Key='{key}' → Shard {shard} → {target_node}"
        self.request_log.append(log)
        return target_node, log
    
    # --- Yaklaşım 2: Smart Client ---
    def smart_client_approach(self, key):
        shard = self.get_shard(key)
        target_node = self.shard_to_node[shard]
        log = f"[Smart Client]  Key='{key}' → Shard {shard} → {target_node} (direkt bağlantı)"
        self.request_log.append(log)
        return target_node, log
    
    # --- Yaklaşım 3: Any Node + Forward ---
    def any_node_approach(self, key, contacted_node="node_0"):
        shard = self.get_shard(key)
        target_node = self.shard_to_node[shard]
        if contacted_node == target_node:
            log = f"[Any Node]      Key='{key}' → {contacted_node} (aynı node, direkt işlem)"
        else:
            log = f"[Any Node]      Key='{key}' → {contacted_node} → forward → {target_node}"
        self.request_log.append(log)
        return target_node, log


router = ShardRouter(num_shards=8)
test_keys = ["user_123", "order_456", "product_789", "session_abc"]

print("=== 3 Request Routing Yaklaşımı Karşılaştırması ===")
print()

for key in test_keys:
    _, log1 = router.routing_tier_approach(key)
    _, log2 = router.smart_client_approach(key)
    _, log3 = router.any_node_approach(key)
    print(log1)
    print(log2)
    print(log3)
    print()

print("\nShard → Node Eşleşmesi (ZooKeeper'da saklanır):")
for shard, node in router.shard_to_node.items():
    print(f"  Shard {shard} → {node}")

## 🗂️ Sharding ve Secondary Indexes

### Sorun

Şimdiye kadar ele aldığımız sharding şemaları, istemcinin erişmek istediği herhangi bir kayıt için **partition key**'i bildiğini varsayar. Bu en kolay, **key-value** veri modelinde gerçekleşir; partition key, birincil anahtarın ilk bölümüdür (veya tamamı), dolayısıyla doğru node'a okuma ve yazma işlemlerini yönlendirmek için partition key kullanılabilir.

**Secondary index** (ikincil dizin) devreye girdiğinde durum karmaşıklaşır. Secondary index genellikle bir kaydı benzersiz olarak tanımlamaz; belirli bir değerin varlığını aramak için kullanılır: kullanıcı 123 tarafından yapılan tüm işlemleri bul, "hogwash" kelimesini içeren tüm makaleleri bul, rengi kırmızı olan tüm arabaları bul vb.

**Key-value store**'lar genellikle secondary index barındırmaz; ancak ilişkisel veritabanlarında standart bir özelliktir, belge veritabanlarında yaygındır. Bu tür indexleme aynı zamanda **Solr** ve **Elasticsearch** gibi tam-metin arama motorlarının var oluş nedenidir.

Secondary index'lerin shard'lara düzgün biçimde eşlenmemesi en büyük zorluktur. Secondary index ile veritabanı shard'lamak için iki temel yaklaşım vardır: **local** ve **global**.

---

## 📍 Local Secondary Indexes (Yerel İkincil Dizinler)

### Temel Fikir

İlk indexleme yaklaşımında her shard, kendi secondary index'ini bağımsız olarak tutar; bu index yalnızca o shard'daki record'ları kapsar. Diğer shard'larda ne saklandığıyla ilgilenmez. Veritabanına yazdığınızda (kayıt ekleme, silme veya güncelleme), yalnızca yazdığınız kaydı içeren shard'ı işlemek yeterlidir. Bu nedenle bu tür secondary index'e **local index** (yerel dizin) denir. Bilgi erişimi bağlamında **document-partitioned index** (belge bölümlü dizin) olarak da bilinir.

### İkinci El Araba Satış Sitesi Örneği

İkinci el araç satan bir web sitesini işlettiğinizi hayal edin. Her ilanın benzersiz bir ID'si var ve sharding için bu ID'yi partition key olarak kullanıyorsunuz (ID 0–499 shard 0'da, 500–999 shard 1'de vb.). Kullanıcıların araçları renge ve markaya göre filtreleyerek arayabilmesi için renk ve marka üzerinde secondary index'lere ihtiyacınız var.

Index tanımlandığında, veritabanı indexlemeyi otomatik gerçekleştirir. Örneğin kırmızı bir araba eklendiğinde, veritabanı shard'ı otomatik olarak kırmızı arabanın ID'sini `color:red` index girişindeki ID listesine ekler. Bu ID listesi aynı zamanda **postings list** (döküman listesi) olarak da adlandırılır.

### Uyarı: Uygulama Tarafında Secondary Index

Veritabanınız yalnızca key-value modelini destekliyorsa, uygulama kodunda değerlerden ID'lere eşleşme oluşturarak secondary index'i kendiniz uygulamak isteyebilirsiniz. Bu yola girerseniz, index'lerin altındaki veriyle tutarlı kalmasını sağlamak için büyük özen göstermeniz gerekir. **Race condition**'lar ve aralıklı yazma hataları (bazı değişikliklerin kaydedilip bazılarının kaydedilmediği durumlar) verinin kolayca senkronizasyonu bozmasına yol açabilir.

### Local Index Okuma

- Aradığınız kaydın **partition key**'ini zaten biliyorsanız, aramayı yalnızca ilgili shard'da yapabilirsiniz.
- Yalnızca bazı sonuçlara ihtiyacınız varsa, isteği herhangi bir shard'a gönderebilirsiniz.
- Ancak tüm sonuçları istiyorsanız ve önceden partition key'lerini bilmiyorsanız, sorguyu **tüm shard'lara** gönderip aldığınız sonuçları birleştirmeniz gerekir; eşleşen kayıtlar tüm shard'lara dağılmış olabilir.

Bu yaklaşıma **scatter/gather** (dağıt/topla) adı verilir.

### Local Index'in Performans Sorunu

Secondary index üzerinde sharded veritabanını sorgulamak bu yaklaşımla oldukça pahalı olabilir. Shard'ları paralel sorgulamanıza rağmen, **tail latency amplification** (kuyruk gecikme büyümesi) riskine açıktır. Ayrıca uygulamanızın ölçeklenebilirliğini kısıtlar: daha fazla shard eklemek daha fazla veri depolamanıza olanak tanır, ancak her shard yine de her sorguyu işlemek zorunda kalıyorsa sorgu throughput'unuzu artırmaz.

### Local Index Kullanan Sistemler

Tüm bu dezavantajlara rağmen local secondary index'ler yaygın olarak kullanılmaktadır:
**MongoDB**, **Riak**, **Cassandra**, **Elasticsearch**, **SolrCloud** ve **VoltDB** local secondary index kullanır.

---

## 🌍 Global Secondary Indexes (Küresel İkincil Dizinler)

### Temel Fikir

Her shard kendi local secondary index'ini tutmak yerine, tüm shard'lardaki verileri kapsayan bir **global index** (küresel dizin) oluşturulabilir. Ancak bu index tek bir node'da saklanamaz; aksi takdirde darboğaz oluşturur ve sharding'in amacını boşa çıkarır. Global index de shard'lanmak zorundadır; fakat primary-key index'inden farklı biçimde shard'lanabilir.

**Örnek:** Tüm shard'lardan kırmızı araçların ID'leri index'te `color:red` altında görünür, ancak index şöyle shard'lanır:
- a ile r arasındaki harflerle başlayan renkler → shard 0
- s ile z arasındaki harflerle başlayan renkler → shard 1

Bu tür index'e **term-partitioned** (terim bölümlü) index de denir. Tam-metin aramada bir *term*, aradığınız metindeki bir anahtar kelimedir. Burada genelleştirilerek secondary index'te arayabileceğiniz herhangi bir değeri ifade eder.

Global index, **term**'i partition key olarak kullanır; belirli bir terimi veya değeri aradığınızda hangi shard'ı sorgulamanız gerektiğini belirleyebilirsiniz. Bir shard, key-range sharding'de olduğu gibi bitişik bir terim aralığı içerebilir ya da terimler hash'lerine göre shard'lara atanabilir.

### Global Index'in Avantajları

Global index'lerin şu avantajı vardır: tek koşullu (örneğin `color = red`) bir sorgu, **postings list**'i almak için yalnızca tek bir shard'dan okumak zorundadır.

Ancak record'ları yalnızca ID'lerini değil içeriklerini de almak istiyorsanız, bu ID'lerden sorumlu tüm shard'lardan okumanız gerekir.

### Birden Fazla Koşul

Birden fazla arama koşulu veya terimi varsa (örneğin belirli renk ve markada araçlar ya da aynı metinde geçen birden fazla kelime için arama), bu terimler büyük olasılıkla farklı shard'lara atanmış olacaktır. İki koşulun mantıksal VE'sini hesaplamak için, sistem her iki **postings list**'te de yer alan tüm ID'leri bulmak zorundadır. Postings list'ler kısaysa sorun yoktur, ancak uzunlarsa bu listeleri ağ üzerinden göndererek kesişimlerini hesaplamak yavaş olabilir.

### Global Index'te Yazma Komplikasyonları

Global secondary index'lerin bir diğer zorluğu, yazmaların local index'e kıyasla daha karmaşık olmasıdır. Tek bir record yazmak, index'in birden fazla shard'ını etkileyebilir (belgedeki her terim farklı bir shard'da olabilir). Bu, secondary index'i altındaki veriyle senkronize tutmayı güçleştirir. Bir seçenek, hem primary record'u saklayan shard'ları hem de secondary index'leri atomik biçimde güncellemek için **distributed transaction** kullanmaktır.

### Global Index Kullanan Sistemler

**CockroachDB**, **TiDB** ve **YugabyteDB** global secondary index kullanır. **DynamoDB** hem local hem global secondary index'i destekler. DynamoDB'de global index'lere yapılan yazmalar **asenkron** olarak yansıtılır; dolayısıyla global index'ten yapılan okumalar **stale** (güncel olmayan) olabilir.

Buna rağmen global index'ler, read throughput write throughput'tan daha yüksekse ve postings list'ler çok uzun değilse faydalıdır.

In [ ]:
# Local vs Global Secondary Index Karşılaştırması
# İkinci el araba satış sitesi örneği

# =============================================
# LOCAL SECONDARY INDEX SİMÜLASYONU
# =============================================

class LocalIndexShard:
    def __init__(self, shard_id, id_range):
        self.shard_id = shard_id
        self.id_range = id_range
        self.records = {}     # ID → car data
        self.color_index = {} # color → list of IDs (local index)
        self.make_index = {}  # make  → list of IDs (local index)
    
    def add_car(self, car_id, color, make):
        self.records[car_id] = {"color": color, "make": make}
        self.color_index.setdefault(color, []).append(car_id)
        self.make_index.setdefault(make, []).append(car_id)
    
    def search_by_color(self, color):
        return self.color_index.get(color, [])


# Veritabanı 2 shard'a bölünmüş
shard0 = LocalIndexShard(0, range(0, 500))
shard1 = LocalIndexShard(1, range(500, 1000))

# Arabalar ekle
shard0.add_car(101, "red",   "Toyota")
shard0.add_car(202, "blue",  "Honda")
shard0.add_car(303, "red",   "Ford")
shard1.add_car(501, "red",   "BMW")
shard1.add_car(602, "green", "Toyota")
shard1.add_car(703, "red",   "Audi")

print("=== LOCAL SECONDARY INDEX ===")
print("'color=red' araması — SCATTER/GATHER gerektirir:")
results_shard0 = shard0.search_by_color("red")
results_shard1 = shard1.search_by_color("red")
print(f"  Shard 0 sonuçları: {results_shard0}")
print(f"  Shard 1 sonuçları: {results_shard1}")
all_results = results_shard0 + results_shard1
print(f"  Birleşik sonuçlar: {all_results}")
print(f"  Sorgulanan shard sayısı: 2 (tüm shard'lar)")

print()

# =============================================
# GLOBAL SECONDARY INDEX SİMÜLASYONU
# =============================================

class GlobalIndex:
    def __init__(self):
        # Index shard'ı: renkler a-m → index_shard_0, n-z → index_shard_1
        self.index_shard_0 = {}  # a-m
        self.index_shard_1 = {}  # n-z
    
    def _get_index_shard(self, color):
        return self.index_shard_0 if color[0] <= 'm' else self.index_shard_1
    
    def add_to_index(self, color, car_id):
        shard = self._get_index_shard(color)
        shard.setdefault(color, []).append(car_id)
    
    def search_by_color(self, color):
        shard = self._get_index_shard(color)
        idx_shard_name = "index_shard_0" if color[0] <= 'm' else "index_shard_1"
        results = shard.get(color, [])
        return results, idx_shard_name


global_index = GlobalIndex()
cars = [
    (101, "red", "Toyota"),
    (202, "blue", "Honda"),
    (303, "red", "Ford"),
    (501, "red", "BMW"),
    (602, "green", "Toyota"),
    (703, "red", "Audi"),
]
for car_id, color, make in cars:
    global_index.add_to_index(color, car_id)

print("=== GLOBAL SECONDARY INDEX ===")
print("'color=red' araması:")
results, idx_shard = global_index.search_by_color("red")
print(f"  Global index shard'ı ({idx_shard}) → {results}")
print(f"  Yalnızca 1 index shard'ı sorgulandı!")
print(f"  Ama record'ları getirmek için birden fazla data shard'ı okunacak")

print()
print("=== KARŞILAŞTIRMA ===")
print("Local Index → Yazma: 1 shard; Okuma: TÜM shard'lar (scatter/gather)")
print("Global Index → Yazma: Birden fazla index shard'ı; Okuma: 1 index shard")

## 📊 Tüm Sharding Yöntemlerinin Karşılaştırması

| Özellik | Key-Range Sharding | Hash Sharding | Hash-Range Sharding |
|---------|-------------------|---------------|---------------------|
| **Range Query Desteği** | ✅ Verimli | ❌ Verimsiz | ⚠️ Kısmi (hash key üzerinde değil) |
| **Hot Spot Riski** | ⚠️ Yüksek (bitişik key'ler) | ✅ Düşük | ✅ Düşük |
| **Shard Sayısı Adaptasyonu** | ✅ Dinamik (split/merge) | ⚠️ Sabit sayıda ise zor | ✅ Dinamik |
| **Rebalancing Maliyeti** | ⚠️ Yüksek (split pahalı) | ✅ Düşük (shard taşıma) | ⚠️ Orta |
| **Uygulayan Sistemler** | HBase, CockroachDB, Bigtable | Citus, Riak, Elasticsearch | Cassandra, DynamoDB, YugabyteDB |

| Özellik | Local Secondary Index | Global Secondary Index |
|---------|----------------------|------------------------|
| **Yazma Karmaşıklığı** | ✅ Basit (1 shard) | ⚠️ Karmaşık (çok shard) |
| **Okuma (Tekil Değer)** | ⚠️ Scatter/Gather (tüm shard'lar) | ✅ Tek shard |
| **Veri Tutarlılığı** | ✅ Anlık | ⚠️ Asenkron olabilir (stale read) |
| **Uygulayan Sistemler** | MongoDB, Cassandra, Elasticsearch | CockroachDB, TiDB, DynamoDB |

## 📝 Bölüm Özeti

Bu bölümde büyük bir dataset'i daha küçük alt kümelere shard'lamanın farklı yollarını inceledik. Sharding, verinin ve işlemlerin tek bir makinede saklanıp işlenmesinin artık mümkün olmadığı durumlarda zorunludur.

### Sharding'in Temel Amacı

Sharding'in amacı, veriyi ve sorgu yükünü birden fazla makineye eşit biçimde yaymak, **hot spot**'lardan (orantısız yüksek yüke sahip node'lar) kaçınmaktır. Bu, veriye uygun bir sharding şeması seçmeyi ve node'lar cluster'a eklenip çıkarıldıkça shard'ları rebalance etmeyi gerektirir.

### İki Temel Sharding Yaklaşımı

#### 1. Key Range Sharding
Key'ler sıralanır; bir shard minimumdan maksimuma tüm key'lere sahiptir. Sıralamanın avantajı, verimli range sorguların mümkün olmasıdır. Ancak uygulama sıralanmış düzende birbirine yakın key'lere sıklıkla erişiyorsa hot spot riski vardır. Bu yaklaşımda shard'lar tipik olarak bir shard çok büyüdüğünde range'i iki alt range'e bölünerek rebalance edilir.

#### 2. Hash Sharding
Her key'e bir hash fonksiyonu uygulanır; bir shard bir hash değerleri aralığını barındırır (ya da başka bir consistent hashing algoritması hash'leri shard'lara eşler). Bu yöntem key'lerin sıralamasını bozarak range sorgularını verimsiz kılar; ancak yükü daha dengeli dağıtabilir. Hash ile shard'larken yaygın yaklaşım, başlangıçta sabit sayıda shard oluşturmak, her node'a birden fazla shard atamak ve node ekleme/çıkarma sırasında shard'ların tamamını bir node'dan diğerine taşımaktır. Key range'lerde olduğu gibi shard split de mümkündür.

### Bileşik Key Yaklaşımı

Key'in ilk bölümünü partition key (yani shard'ı belirleyecek kısım) olarak kullanmak ve record'ları o shard içinde key'in geri kalanına göre sıralamak yaygın bir yaklaşımdır. Bu sayede aynı partition key'e sahip record'lar arasında hâlâ verimli range sorguları yapılabilir.

### Request Routing ve Koordinasyon

Sorguları uygun shard'a yönlendirme tekniklerini ve shard'ların node'lara atanmasını takip etmek için koordinasyon servisinin sıklıkla nasıl kullanıldığını inceledik.

### Secondary Index ve Sharding

Son olarak sharding ile secondary index arasındaki etkileşimi ele aldık. Secondary index'in de shard'lanması gerekir. İki yöntem vardır:

- **Local secondary indexes:** Secondary index'ler primary key ve değerle aynı shard'da saklanır. Yazma sırasında yalnızca tek bir shard güncellenmesi gerekir; ancak secondary index araması tüm shard'lardan okuma gerektirir.

- **Global secondary indexes:** Secondary index'ler, indexlenen değerlere göre ayrı biçimde shard'lanır. Secondary index'teki bir girdi, primary key'in tüm shard'larından record'lara başvurabilir. Bir record yazıldığında birden fazla secondary index shard'ının güncellenmesi gerekebilir; ancak postings list'in okunması tek bir shard'dan karşılanabilir (gerçek record'ların getirilmesi için yine de birden fazla shard'dan okuma gerekir).

### Tek Node Bağımsızlığı ve Çok Shard'lı Yazma Sorunu

Tasarım gereği her shard büyük ölçüde bağımsız çalışır; bu, sharded bir veritabanının birden fazla makineye ölçeklenmesini sağlar. Ancak birden fazla shard'a yazması gereken işlemler sorunludur: bir shard'a yazma başarılıyken diğeri başarısız olursa ne olur? Bu soru sonraki bölümlerde ele alınacaktır.

## 📖 Terimler Sözlüğü

| Terim | Türkçe Açıklama |
|-------|----------------|
| **Sharding** | Büyük bir dataset'i birden fazla node'a dağıtmak için daha küçük parçalara (shard) bölme işlemi |
| **Partition** | Sharding ile eş anlamlı kullanılır; bazı sistemlerde farklı anlam taşır |
| **Shard** | Daha büyük bir dataset'in parçası; farklı sistemlerde tablet, region, vnode gibi isimler alır |
| **Partition Key** | Bir kaydın hangi shard'a gideceğini belirleyen sütun veya alan |
| **Hot Spot** | Orantısız yüksek yük taşıyan shard veya node |
| **Hot Key** | Çok fazla istek alan tekil bir key |
| **Skew** | Shard'lar arasında dengesiz veri veya yük dağılımı |
| **Rebalancing** | Shard'ların node'lar arasında yeniden dağıtılması işlemi |
| **Key Range Sharding** | Her shard'a belirli bir key aralığı atayan sharding yöntemi |
| **Hash Sharding** | Shard atamak için hash fonksiyonu kullanan sharding yöntemi |
| **Consistent Hashing** | Shard sayısı değiştiğinde minimum veri taşınmasını sağlayan hash algoritmalar ailesi |
| **Rendezvous Hashing** | Consistent hashing varyantı; highest random weight olarak da bilinir |
| **Pre-splitting** | Veritabanı boşken başlangıç shard kümesi yapılandırma işlemi |
| **Request Routing** | Bir key için doğru node'u bulma sorunu |
| **Routing Tier** | Shard atamasını bilen ve istekleri yönlendiren ayrı bir bileşen |
| **Service Discovery** | Sistemdeki servisleri bulma mekanizması |
| **ZooKeeper** | Shard atamalarını izleyen yaygın koordinasyon servisi |
| **etcd** | Dağıtık key-value store; koordinasyon için kullanılır (Kubernetes'te kullanılır) |
| **Raft** | Dağıtık consensus protokolü; coordinator election için kullanılır |
| **Gossip Protocol** | Node'lar arasında zayıf tutarlılıkla durum yayan iletişim protokolü |
| **Split Brain** | İki bileşenin çelişkili kararlar aldığı senaryo |
| **Local Secondary Index** | Her shard'ın kendi record'larını kapsayan kendi secondary index'ini tutması |
| **Global Secondary Index** | Tüm shard'lardaki verileri kapsayan ve ayrıca shard'lanan index |
| **Scatter/Gather** | Sorguyu tüm shard'lara gönderip sonuçları birleştirme yaklaşımı |
| **Postings List** | Bir index girişi için ID'lerin listesi |
| **Term-Partitioned Index** | Term'in partition key olarak kullanıldığı global index türü |
| **Distributed Transaction** | Birden fazla node üzerinde atomiklik gerektiren işlem |
| **Multitenancy** | Birden fazla müşterinin (tenant) aynı altyapıyı paylaşması |
| **Cell-Based Architecture** | Tenant gruplarının bağımsız hücrelerde organize edildiği mimari |
| **Fault Isolation** | Bir cell'deki arızanın diğer cell'leri etkilemesini önleme |
| **NUMA** | Non-Uniform Memory Access; bazı bellek bankalarının bazı CPU'lara daha yakın olduğu mimari |
| **Cascading Failure** | Bir arızanın diğerlerini tetikleyerek yayılan zincir arıza |
| **Data Residency** | Verinin belirli coğrafi konumlarda saklanması zorunluluğu |
| **Schema Migration** | Veritabanı şemasını güncelleme işlemi |
| **Read Scaling** | Okuma yükünü birden fazla replica'ya dağıtarak ölçeklendirme |
| **Horizontal Scaling** | Daha küçük makineler ekleyerek kapasiteyi artırma (scale-out) |
| **Vertical Scaling** | Tek bir makineyi büyüterek kapasiteyi artırma (scale-up) |

## 🔗 Serinin Diğer Bölümleri

Bu notebook, DDIA kitabının Türkçe açıklamalar serisinin bir parçasıdır:

- Bölüm 3: Data Models and Query Languages
- **Bölüm 7: Sharding ← (Bu bölüm)**

---

## 📚 Kaynaklar ve İleri Okuma

Bu bölümde referans verilen bazı önemli kaynaklar:

- **Consistent Hashing:** Karger et al., *Consistent Hashing and Random Trees*, STOC 1997
- **Amazon DynamoDB:** Elhemali et al., *Amazon DynamoDB: A Scalable, Predictably Performant, and Fully Managed NoSQL Database Service*, USENIX ATC 2022
- **FoundationDB:** Zhou et al., *FoundationDB: A Distributed Unbundled Transactional Key Value Store*, SIGMOD 2021
- **Shard Manager:** Lee et al., *Shard Manager: A Generic Shard Management Framework*, SOSP 2021

---

*Designing Data-Intensive Applications (Martin Kleppmann) kitabından Türkçe notlar.*  
*Tüm teknik terimler orijinal İngilizce halleriyle korunmuştur.*